https://github.com/tljcpa/QLoRA_Tutorial?tab=readme-ov-file#qwen25-3b-qlora-%E5%BE%AE%E8%B0%83%E5%85%A8%E6%B5%81%E7%A8%8B%E6%95%99%E7%A8%8B

# 第一阶段:QLoRA 训练




> 注意事项（训练前必读）

1.   运行前先挂载 Google Drive，所有产物保存到 Drive，防止断线丢失。
2.   确保你的数据文件已上传到 Drive 的 data/ 目录。
3. 第一次运行建议用少量数据（100 条以内）验证流程跑通，再换完整数据集。
4. 训练过程中不要关闭浏览器标签页，Colab 会话会断开。
5. 训练结束后，不要在同一个会话里直接做第二阶段，必须重启会话释放显存。








## Cell 1:环境初始化

作用：挂载 Drive，安装依赖，创建目录结构。每次新会话开始时运行一次。

In [2]:
# 挂载 Google Drive ， 防止断线后输出消失
from google.colab import drive
drive.mount('/content/drive')

# 安装所有核心依赖（-q 表示静默模式，-U 表示安装最新版）
!pip install -q -U transformers peft bitsandbytes accelerate datasets trl tensorboard

# 创建项目目录结构
import os
PROJECT_DIR = '/content/drive/MyDrive/Qwen-QLoRA'
DATA_DIR = os.path.join(PROJECT_DIR, 'data') # 数据文件目录
OUTPUT_DIR = os.path.join(PROJECT_DIR, 'output') # 输出文件目录
MODEL_DIR   = os.path.join(PROJECT_DIR, 'model') # 模型缓存目录
for d in [DATA_DIR, OUTPUT_DIR]:
    os.makedirs(d, exist_ok=True)
print(f'环境就绪，工程主目录：{PROJECT_DIR}')

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 74.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 526.8/526.8 kB 28.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.0/531.0 kB 31.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 117.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 21.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.19.0 requires tensorboard~=2.19.0, but you have tensorboard 2.20.0 which is incompatible.
环境就绪，工程主目录：/content/drive/MyDrive/Qwen-QLoRA


## Cell 2:训练主程序
作用：加载模型、注入 LoRA、加载数据、执行训练、保存 Adapter。

### 第一块:清理环境 + 全局配置

这是每次换项目时唯一需要修改的区域，其他代码不用动。

In [ ]:
from huggingface_hub import snapshot_download

MODEL_DIR = '/content/drive/MyDrive/Qwen-QLoRA/model'

# 检查是否已经下载过
if os.path.exists(os.path.join(MODEL_DIR, 'config.json')):
    print('模型已存在，跳过下载')
else:
    print('开始下载，约 6GB，需要 3~5 分钟，请保持页面...')
    snapshot_download(
        repo_id='Qwen/Qwen2.5-3B-Instruct',
        local_dir=MODEL_DIR,
    )
    print('模型下载完成，已保存到 Drive')

模型已存在，跳过下载


In [ ]:
import os, torch, torch.nn as nn, bitsandbytes as bnb
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

# [不用改] 清理可能导致 bf16 报错的环境变量（T4 不支持 bf16）
if 'ACCELERATE_MIXED_PRECISION' in os.environ:
    del os.environ['ACCELERATE_MIXED_PRECISION']

# ==================== 全局配置区（每次换项目只改这里）====================
CFG = {
    'model_id' : '/content/drive/MyDrive/Qwen-QLoRA/model', # [必须改] 换模型时改这里
    # 'dataset_path' : '/content/drive/MyDrive/Qwen-QLoRA/data/train.jsonl', # [必须改] 你的数据文件路径
    'output_dir' : '/content/drive/MyDrive/Qwen-QLoRA/output', # [必须改] 输出目录
    'epochs' : 3, # [可选改] 训练轮数，数据少时用 1~2，数据多时用 3~5
    'lr' : 1e-4, # [可选改] 学习率，LoRA 微调通常 1e-4 到 5e-4
    'r' : 16, # [可选改] LoRA rank，数据少用 8，数据多用 32
    'alpha' : 32, # [可选改] 始终保持 alpha = 2 × r
    'dropout' : 0.05 # [可选改] 过拟合严重时加大到 0.1，数据多时设 0
}

# [不用改] T4 显存限制，batch_size 只能是 1，用梯度累积补偿
# 等效 batch_size = 1 × 16 = 16
bs, grad_accum = 1, 16

### 第二块:加载Tokenizer和模型

In [ ]:
# [不用改] 加载 Tokenizer
tokenizer = AutoTokenizer.from_pretrained(CFG['model_id'], trust_remote_code=True)
# pad_token 用于填充不同长度的样本到同一 batch，LLaMA/Qwen 默认没有，用 eos_token 代替
tokenizer.pad_token = tokenizer.eos_token

# [不用改] 4bit 量化配置
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, # 用 4bit 加载，显存减少 4 倍
    bnb_4bit_quant_type='nf4', # NF4 量化，误差最小的 4bit 方式
    bnb_4bit_compute_dtype=torch.float16, # 存储 4bit，计算时临时解压成 fp16
    bnb_4bit_use_double_quant=True # 双重量化，再省约 200~300MB 显存
)

# [不用改] 加载基座模型
model = AutoModelForCausalLM.from_pretrained(
    CFG['model_id'],
    quantization_config=bnb_config,
    device_map='auto', # 自动分配层到 GPU/CPU
    trust_remote_code=True
)

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

### 第三块:稳定性配置（三行必须写）

In [ ]:
# [不用改] 关闭推理时的 KV Cache，训练时用不到，开着白占显存
model.config.use_cache = False

# [不用改] 梯度检查点：不保存所有中间激活值，用重新计算换显存
# 代价：训练速度慢约 20%，收益：显存减少约 50%，T4 上必须开
model.gradient_checkpointing_enable()

# [不用改] 4bit 量化后梯度追踪被打断，这行恢复它
# 不加这行 LoRA 参数不会被训练，Loss 不下降但你不知道原因
model.enable_input_require_grads()

# [不用改] 为 4bit 训练做进一步稳定性处理
model = prepare_model_for_kbit_training(model)

### 第四块:注入 LoRA

In [ ]:
# [不用改] 自动嗅探模型里所有线性层的名字
# 这样写的原因：不同模型层名字不同（有的叫 q_proj，有的叫 query）
# 硬编码层名换个模型就报错，动态嗅探对任何模型都通用
cls_types = (bnb.nn.Linear4bit, bnb.nn.Linear8bitLt, nn.Linear)
target_modules = list({
    name.split('.')[-1]
    for name, module in model.named_modules()
    if isinstance(module, cls_types) and name.split('.')[-1] != 'lm_head'  # 排除输出层，风险高
})

# [不用改，参数在 CFG 里改] 将 LoRA 注入模型
model = get_peft_model(model, LoraConfig(
    r=CFG['r'],
    lora_alpha=CFG['alpha'],
    target_modules=target_modules,
    lora_dropout=CFG['dropout'],
    bias='none', # 不训练 bias，省参数，效果无明显差异
    task_type='CAUSAL_LM'
))

# 打印可训练参数占比，正常应该在 0.5%~2% 左右
model.print_trainable_parameters()

trainable params: 29,933,568 || all params: 3,115,872,256 || trainable%: 0.9607


### 第五块:数据加载与格式化


> 数据格式要求


> 你的 train.jsonl 文件每一行必须是一个 JSON 对象，包含三个字段：


*   instruction：任务描述（必填）
*   input：附加输入内容（没有时填空字符串 ""）
* output：期望模型输出的答案（必填）







```
{"instruction": "帮我写一个排序函数", "input": "", "output": "def sort(lst): return sorted(lst)"}
{"instruction": "翻译成英文", "input": "今天天气很好", "output": "The weather is nice today"}
```





In [ ]:
# [必须改] 数据加载
# 如果是 jsonl 或 json 文件：
# dataset = load_dataset('json', data_files=CFG['dataset_path'], split='train')
dataset = load_dataset('tatsu-lab/alpaca', split='train[:500]')
# 如果是 csv 文件，把 'json' 改成 'csv'
# 如果是 HuggingFace 上的数据集，改成：
# dataset = load_dataset('数据集名字', split='train')

# [必须改] 格式化函数：把你的数据转成 Qwen 能理解的对话格式
# <|im_start|> 和 <|im_end|> 是 Qwen 系列的特殊分隔符，不要改动
def format_prompts(examples):
    texts = []
    for inst, inp, out in zip(
        examples['instruction'], # [必须改] 换成你的字段名
        examples['input'], # [必须改] 换成你的字段名，没有这个字段就删掉这行
        examples['output'] # [必须改] 换成你的字段名
    ):
        # 如果你的数据没有 input 字段，把 f'{inst}\n{inp}' 改成 f'{inst}'
        text = f'<|im_start|>user\n{inst}\n{inp}<|im_end|>\n<|im_start|>assistant\n{out}<|im_end|>'
        texts.append(text)
    return {'text': texts}

train_dataset = dataset.map(format_prompts, batched=True)
print(f'数据集大小：{len(train_dataset)} 条')
print('示例：\n', train_dataset[0]['text'][:10])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


数据集大小：500 条
示例：
 <|im_start




> 如果你的数据字段名不叫 instruction/input/output,只需要改 format_prompts 函数里 zip() 括号内的字段名，其他代码完全不动。




> 例如字段名是 question 和 answer：


```
for q, a in zip(examples['question'], examples['answer']):
    text = f'<|im_start|>user\n{q}<|im_end|>\n<|im_start|>assistant\n{a}<|im_end|>'
```



### 第六块:训练参数与执行

In [ ]:
import os
os.makedirs(CFG['output_dir'], exist_ok=True)

# [不用改] 训练参数（已针对 T4 调优，不要随意修改）
training_args = SFTConfig(
    output_dir=CFG['output_dir'],
    per_device_train_batch_size=bs, # T4 只能是 1，否则 OOM
    gradient_accumulation_steps=grad_accum, # 等效 batch_size=16
    optim='paged_adamw_8bit',  # 8bit 优化器，内存不够时自动换页到 CPU
    logging_steps=10, # 每 10 步打印一次 Loss
    save_strategy='epoch', # 每个 epoch 保存一次 checkpoint
    learning_rate=CFG['lr'],
    fp16=False, # 必须 False：T4 不支持 bf16，fp16=True 会触发 bf16 代码路径导致崩溃
    bf16=False, # 必须 False：同上
    num_train_epochs=CFG['epochs'],
    dataset_text_field='text', # 告诉 trainer 用哪个字段
    max_length=512,                         # 超过 512 token 的样本会被截断
    packing=False                           # 不把多个短样本打包成一个，避免上下文混乱
)

# [不用改] 创建训练器并启动
trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    args=training_args
)

print('开始训练...')
trainer.train()

# [不用改] 保存 LoRA Adapter（只有几十 MB，不是完整模型）
adapter_path = os.path.join(CFG['output_dir'], 'lora-adapter')
trainer.model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)
print(f'Adapter 已保存至: {adapter_path}')

开始训练...


Step,Training Loss


Step,Training Loss


KeyboardInterrupt: 

## 如何应对 Colab 断线（断点续训）


> Colab 免费层随时可能断线。由于我们配置了 save_strategy='epoch'，每个 epoch 结束时都会自动保存 checkpoint 到 output_dir。


> 断线重连后，重新运行 Cell 1（环境初始化）和 Cell 2 的前五块（不要执行 trainer.train()），然后把最后的训练执行部分改成：


```
# [不用改] 从最近一次 checkpoint 恢复训练，自动找到 output_dir 下最新的 checkpoint
print('开始训练（从断点恢复）...')
trainer.train(resume_from_checkpoint=True)
```







# 第二阶段:权重合并


> 强制要求：运行第二阶段前必须重启会话




> 点击顶部菜单：代码执行程序 -> 重新启动会话



> 原因：训练后 GPU 显存和 RAM 都几乎被占满。合并操作需要在 CPU 上加载完整的 float16 模型（约 6GB），内存不释放直接运行必然 OOM。



> 重启后先运行一次：!pip install -q -U transformers peft
然后再运行下方代码。





### 为什么合并是必要的



> 训练完成后你有两样东西：原始基座模型（6GB）和 LoRA Adapter（几十 MB）。第三阶段的 llama.cpp 只接受一个完整的模型文件，不理解「基座 + 补丁」这种分离格式，所以必须先合并。


> 合并的数学本质：对每个被 LoRA 修改的层，计算 W_merged = W + (alpha/r) × A × B，然后删除 A 和 B，留下普通的 W_merged。这就是 merge_and_unload() 的含义：merge（合并）+ unload（卸载 LoRA 结构）。





## Cell 3:防 OOM 权重合并

In [ ]:
import os, gc, torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

# ==================== 路径配置（只改这三行）====================
BASE_MODEL = '/content/drive/MyDrive/Qwen-QLoRA/model' # [必须改] 你用的基座模型
ADAPTER_DIR = '/content/drive/MyDrive/Qwen-QLoRA/output/lora-adapter' # [必须改] 第一阶段保存的 Adapter 路径
MERGED_DIR = '/content/drive/MyDrive/Qwen-QLoRA/output/merged-fp16'  # [必须改] 合并后的模型保存路径

# [不用改] 临时 offload 目录放在本地盘（比 Drive 快得多）
OFFLOAD_DIR = '/content/offload_cache'
os.makedirs(OFFLOAD_DIR, exist_ok=True)

# [不用改] 以 CPU Offload 模式加载基座模型
# device_map={'': 'cpu'}：强制所有层放 CPU，不碰 GPU
# low_cpu_mem_usage=True：按层逐一加载，防止整个文件同时在内存里
# offload_folder：内存不够时把暂时用不到的层换页到磁盘
# offload_state_dict=True：把 state dict 本身也换页到磁盘，进一步压低内存峰值
print('1. 加载基座模型（按层加载，约需 1~3 分钟）...')
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map={'': 'cpu'},
    low_cpu_mem_usage=True,
    offload_folder=OFFLOAD_DIR,
    offload_state_dict=True, # 把 state dict 换页到磁盘，对内存吃紧的环境有帮助
    trust_remote_code=True
)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

# [不用改] 挂载 Adapter 并执行合并
# PeftModel.from_pretrained 把 A、B 矩阵附着到基座模型对应的层旁边
# merge_and_unload 执行 W = W + (alpha/r)×A×B，然后删除 A 和 B
print('2. 挂载 Adapter 并合并权重...')
peft_model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
merged_model = peft_model.merge_and_unload()

# [不用改] 强制释放旧对象内存
# del 只删除变量名，gc.collect() 才真正回收内存
# 必须同时 del 两个变量，因为 peft_model 内部持有对 base_model 的引用
# 不做这步，保存时内存里同时存在旧对象+序列化数据，必然 OOM
print('3. 释放旧对象内存...')
del base_model
del peft_model
gc.collect()

# [不用改] 分片保存合并后的模型
# max_shard_size='1GB'：每次只序列化 1GB，避免整个 6GB 同时在内存里（需要额外 6GB）
# safe_serialization=True：使用更安全的 safetensors 格式
print('4. 分片保存模型（约需 3~5 分钟）...')
merged_model.save_pretrained(
    MERGED_DIR,
    safe_serialization=True,
    max_shard_size='1GB'
)
tokenizer.save_pretrained(MERGED_DIR)
print(f'合并完成！模型位于: {MERGED_DIR}')

`torch_dtype` is deprecated! Use `dtype` instead!


1. 加载基座模型（按层加载，约需 1~3 分钟）...


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

2. 挂载 Adapter 并合并权重...
3. 释放旧对象内存...
4. 分片保存模型（约需 3~5 分钟）...


Writing model shards:   0%|          | 0/7 [00:00<?, ?it/s]

合并完成！模型位于: /content/drive/MyDrive/Qwen-QLoRA/output/merged-fp16


In [ ]:
import json, os

special_tokens_map = {
    "bos_token": "<|endoftext|>",
    "eos_token": "<|im_end|>",
    "pad_token": "<|endoftext|>",
    "unk_token": "<|endoftext|>",
    "additional_special_tokens": [
        "<|im_start|>",
        "<|im_end|>"
    ]
}

MERGED_DIR = '/content/drive/MyDrive/Qwen-QLoRA/output/merged-fp16'
path = os.path.join(MERGED_DIR, 'special_tokens_map.json')
with open(path, 'w') as f:
    json.dump(special_tokens_map, f, indent=2)

print('✅ special_tokens_map.json 已生成')

✅ special_tokens_map.json 已生成


### 合并成功后的目录结构




```
merged-fp16/
├── model-00001-of-00006.safetensors <- 模型权重分片（每块约 1GB）
├── model-00002-of-00006.safetensors
├── ...
├── model.safetensors.index.json <- 索引文件，记录每层在哪个分片里
├── config.json <- 模型结构配置
├── tokenizer.json <- 词表
├── tokenizer_config.json <- tokenizer 配置
└── special_tokens_map.json <- 特殊 token 映射
```





# 第三阶段:GGUF 量化

### 为什么需要这一步


> 合并后的模型是 float16 格式，大小约 6GB，需要显卡才能推理。GGUF 量化做了两件事：

> 第一：把 float16（每参数 16bit）压缩成 Q4_K_M（每参数约 4bit），文件从 6GB 缩小到约 1.8GB。


> 第二：生成的 GGUF 格式专为 CPU 推理优化，配合 llama.cpp 可以在没有显卡的普通电脑上运行，速度比 Python 框架快 5~10 倍。


> 为什么是两步（先转 FP16 GGUF，再量化到 Q4）？ 因为 llama.cpp 的量化工具只接受 GGUF 格式作为输入，必须先把 HuggingFace 格式转成 GGUF，再做精度压缩。









## Cell 4:GGUF 量化转换



> 这个 Cell 用 %%bash 运行，整个 Cell 是一个 shell 脚本，不是 Python。不需要重启会话，接着第二阶段的会话直接运行即可。



In [ ]:
%%bash

# ==================== 路径配置（只改这里）====================
STABLE_COMMIT='b4600'  # [可选改] llama.cpp 的稳定版本号，如果这个版本将来出问题再换
MERGED_MODEL_PATH='/content/drive/MyDrive/Qwen-QLoRA/output/merged-fp16'  # [必须改]
GGUF_DIR='/content/drive/MyDrive/Qwen-QLoRA/output/gguf'                   # [必须改]
GGUF_FP16="$GGUF_DIR/qwen2.5-3b-fp16.gguf"    # [可选改] 中间产物文件名
GGUF_Q4="$GGUF_DIR/qwen2.5-3b-q4_k_m.gguf"   # [可选改] 最终产物文件名
# =============================================================

# 创建输出目录；rm -rf 删除旧的 llama.cpp（防止 clone 时因目录已存在而报错）
mkdir -p $GGUF_DIR
rm -rf /content/llama.cpp

echo '1. 克隆 llama.cpp 并切换到稳定版本...'
cd /content
git clone https://github.com/ggerganov/llama.cpp.git
cd llama.cpp
git checkout $STABLE_COMMIT   # 锁定版本，防止新版本 API 变化导致脚本失效
pip install -q gguf            # convert_hf_to_gguf.py 的 Python 依赖

echo '2. 编译量化工具（CMake 方式，Makefile 已废弃）...'
# cmake -B build：配置阶段，检查编译器和依赖，结果放到 build 目录
# cmake --build build：编译阶段
# --config Release：开启优化，比 Debug 快 3~10 倍
# -j：并行编译，用所有 CPU 核心
# --target llama-quantize：只编译量化工具，不编译其他（节省时间）
cmake -B build && cmake --build build --config Release -j --target llama-quantize

echo '3. 修复 tokenizer_config.json（覆盖 save_pretrained 的已知 Bug）...'
# HuggingFace 的 save_pretrained 有时会把 special tokens 从字典写成列表
# llama.cpp 转换脚本读取时期望字典，看到列表就报错
# 解法：直接从官方仓库下载原版文件覆盖
# [必须改] 如果你换了模型，把 URL 里的模型名改掉
wget -q -O $MERGED_MODEL_PATH/tokenizer_config.json \
  https://huggingface.co/Qwen/Qwen2.5-3B-Instruct/resolve/main/tokenizer_config.json

echo '4. 转换 HuggingFace 格式 -> FP16 GGUF...'
# 必须先转成 fp16 GGUF，才能再量化成 Q4
python convert_hf_to_gguf.py $MERGED_MODEL_PATH --outfile $GGUF_FP16 --outtype f16

echo '5. 执行 Q4_K_M 量化...'
# find 动态查找编译产物路径（不同系统路径不同，动态查找更稳）
QUANTIZE_BIN=$(find build -name llama-quantize -type f | head -n 1)
# 参数顺序：输入文件 输出文件 量化方式
# Q4_K_M：4bit 量化，K-Quant 算法，Medium 精度，性价比最高的方案
$QUANTIZE_BIN $GGUF_FP16 $GGUF_Q4 Q4_K_M

echo "全流程完成！最终模型位于: $GGUF_Q4"

Process is terminated.


### 量化完成后如何使用


> 方式一（推荐新手）：下载 LM Studio，把 .gguf 文件拖进去，直接图形界面聊天。

> 方式二（服务器部署）：使用 llama.cpp 的 llama-cli 或 llama-server 命令行工具直接推理。



